# Trading Summary

### IMPORTS

In [ ]:
# YEARS = [2019, 2020, 2021]

In [4]:
import sys, os
sys.path.insert(0, "/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.modules.pop("src", None)
os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")

In [5]:
import pandas as pd
import numpy as np
from math import sqrt, pi, exp
from arch import arch_model
import yfinance as yf 
from scipy.optimize import minimize
import warnings
from arch.__future__ import reindexing
from arch.utility.exceptions import ConvergenceWarning
from src.msGarch import msGARCH
from src.metrics import metrics
from src.msGarch import msGARCHIV
from tqdm import tqdm

from xgboost import XGBRegressor
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="y is poorly scaled")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [16]:
# import sys
# !{sys.executable} -m pip install xgboost

In [17]:
# # DO NOT RUN UNTIL CHANGES TO DEPENDENCIES 
# pip freeze > requirements.txt

### 19-21 01_03 Trading Summary

In [11]:
import pickle

with open("../strategy_outputs/2019_2022/07_30/backtest_results_2019.pkl", "rb") as f:
# with open("../strategy_outputs/2019_2022/07_30/backtest_results_2019.pkl", "rb") as f:
    backtest_results_2019 = pickle.load(f)

# with open("../msg_mfiv200/strategy_outputs/2019_22/07_30/backtest_results_2020.pkl", "rb") as f:
with open("../strategy_outputs/2019_2022/07_30/backtest_results_2020.pkl", "rb") as f:
    backtest_results_2020 = pickle.load(f)

# with open("../msg_mfiv200/strategy_outputs/2019_22/07_30/backtest_results_2021.pkl", "rb") as f:
with open("../strategy_outputs/2019_2022/07_30/backtest_results_2021.pkl", "rb") as f:
    backtest_results_2021 = pickle.load(f)

globals().update(backtest_results_2019)
globals().update(backtest_results_2020)
globals().update(backtest_results_2021)

In [12]:
models_to_test = [
    "ms_LSTM_wIV",
    "ms_LSTM_noIV",
    "ms_base",
    "garch",
    "xg_boost",
    "tarch",
    "midas",
    "sv",
    "gjr",
    "egarch"
]

In [13]:
import numpy as np
import pandas as pd

def perf_stats(equity, periods_per_year=365*24, rf=0.0):
    eq = pd.Series(equity, dtype=float).dropna()

    if len(eq) < 2:
        return {
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "MaxDrawdown": np.nan,
            "Calmar": np.nan,
            "CAGR": np.nan,
            "Volatility": np.nan,
            "TotalReturn": np.nan
        }

    ret = eq.pct_change().dropna()
    if len(ret) == 0:
        return {
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "MaxDrawdown": np.nan,
            "Calmar": np.nan,
            "CAGR": np.nan,
            "Volatility": np.nan,
            "TotalReturn": np.nan
        }

    rf_per_period = (1 + rf) ** (1 / periods_per_year) - 1
    excess_ret = ret - rf_per_period

    mean_excess = excess_ret.mean()
    vol = ret.std(ddof=1)
    downside = excess_ret[excess_ret < 0]
    downside_vol = downside.std(ddof=1)

    sharpe = (mean_excess / vol) * np.sqrt(periods_per_year) if vol > 0 else np.nan
    sortino = (mean_excess / downside_vol) * np.sqrt(periods_per_year) if downside_vol > 0 else np.nan

    cummax = eq.cummax()
    drawdown = eq / cummax - 1
    max_dd = drawdown.min()

    total_return = eq.iloc[-1] / eq.iloc[0] - 1
    years = len(ret) / periods_per_year
    cagr = (eq.iloc[-1] / eq.iloc[0]) ** (1 / years) - 1 if years > 0 else np.nan

    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan
    ann_vol = vol * np.sqrt(periods_per_year)

    return {
        "Sharpe": sharpe,
        "Sortino": sortino,
        "MaxDrawdown": max_dd,
        "Calmar": calmar,
        "CAGR": cagr,
        "Volatility": ann_vol,
        "TotalReturn": total_return
    }

In [14]:
import pandas as pd

rows = []

for y in [2019, 2020, 2021]:
    for q in ["q1","q2","q3","q4"]:
        for model in models_to_test:
            var_name = f"{model}_{str(y)[2:]}_{q}"
            
            if var_name in globals():
                data = globals()[var_name]
                
                try:
                    equity = data["equity"]
                except:
                    equity = [x[0] for x in data]
                
                stats = perf_stats(equity)
                stats["year"] = y
                stats["quarter"] = q
                stats["model"] = model
                
                rows.append(stats)

df_metrics = pd.DataFrame(rows)


In [15]:
metric_tables = {}

for metric in ["Sharpe", "Sortino", "Calmar", "MaxDrawdown", "TotalReturn"]:
    metric_tables[metric] = (
        df_metrics
        .pivot(index="model", columns=["year", "quarter"], values=metric)
        .round(2)
    )

metric_tables

{'Sharpe': year          2019                    2020                    2021        \
 quarter         q1    q2    q3    q4    q1    q2    q3    q4    q1    q2   
 model                                                                      
 egarch        0.52 -0.24  0.54 -0.73 -1.71  1.04  1.04 -0.69 -2.41  2.24   
 garch         0.64 -0.20  0.09 -0.73 -1.72  0.50  0.93 -1.31 -2.94  1.85   
 gjr           0.52 -0.24  0.54 -0.73 -1.71  1.04  1.04 -0.69 -2.41  2.24   
 midas         0.52 -0.24  0.42 -0.73 -1.27  1.03  1.27 -0.04  1.45  2.30   
 ms_LSTM_noIV  0.52 -0.24  0.42 -0.81 -1.76  0.64  1.06 -0.71  2.41  3.60   
 ms_LSTM_wIV   0.49 -0.29  0.34 -1.03 -1.20  1.47  1.08 -0.46  2.48  1.85   
 ms_base       0.52 -0.24  0.41 -0.81 -1.80  1.05  1.12 -0.70  2.42  2.01   
 sv            0.34  0.23  0.64 -0.77 -1.42  0.47  0.45 -0.23  2.19  1.75   
 tarch         0.52 -0.24  0.54 -0.73 -1.71  1.04  1.04 -0.69 -2.41  2.24   
 xg_boost      0.52 -0.24  0.54 -0.73 -0.74  1.04  1.04 -0.27 -2.3

In [16]:
with open("sharpe_table.tex", "w") as f:
    f.write(metric_tables["Sharpe"].to_latex(float_format="%.2f"))

In [17]:
df_metrics_cum = (
    df_metrics
    .groupby(["model"], as_index=False)[["Sharpe", "Sortino", "Calmar", "MaxDrawdown", "TotalReturn"]]
    .mean()
    .round(2)
)

df_metrics_cum

,model,Sharpe,Sortino,Calmar,MaxDrawdown,TotalReturn
0,egarch,0.33,0.47,6.17,-0.36,0.03
1,garch,0.12,0.27,5.13,-0.34,0.04
2,gjr,0.33,0.47,6.17,-0.36,0.03
3,midas,0.74,0.86,4.10,-0.34,0.07
4,ms_LSTM_noIV,0.81,1.44,7.55,-0.35,0.08
5,ms_LSTM_wIV,0.75,0.84,4.37,-0.30,0.11
6,ms_base,0.70,1.25,7.00,-0.35,0.07
7,sv,0.62,0.56,4.04,-0.28,0.15
8,tarch,0.33,0.47,6.17,-0.36,0.03
9,xg_boost,0.45,0.54,6.26,-0.36,0.05


In [18]:
with open("cumulative_metrics_table.tex", "w") as f:
    f.write(df_metrics_cum.to_latex(index=False, float_format="%.2f"))

### 22_24 01_03 Trading Summary

In [ ]:
import pickle

# with open("../msg_mfiv200/strategy_outputs/2022_25/07_30/backtest_results_2022.pkl", "rb") as f:
with open("../strategy_outputs/2022_2025/07_30/backtest_results_2022.pkl", "rb") as f:
    backtest_results_2022 = pickle.load(f)

# with open("../msg_mfiv200/strategy_outputs/2022_25/07_30//backtest_results_2023.pkl", "rb") as f:
with open("../strategy_outputs/2022_2025/07_30//backtest_results_2023.pkl", "rb") as f:
    backtest_results_2023 = pickle.load(f)

# with open("../msg_mfiv200/strategy_outputs/2022_25/07_30/backtest_results_2024.pkl", "rb") as f:
with open("../strategy_outputs/2022_2025/07_30/backtest_results_2024.pkl", "rb") as f:
    backtest_results_2024 = pickle.load(f)


globals().update(backtest_results_2022)
globals().update(backtest_results_2023)
globals().update(backtest_results_2024)

In [48]:
models_to_test = [
    "ms_LSTM_wIV",
    "ms_LSTM_noIV",
    "ms_base",
    "garch",
    "xg_boost",
    "tarch",
    "midas",
    "sv",
    "gjr",
    "egarch"
]

In [49]:
import numpy as np
import pandas as pd

def perf_stats(equity, periods_per_year=365*24, rf=0.0):
    eq = pd.Series(equity, dtype=float).dropna()

    if len(eq) < 2:
        return {
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "MaxDrawdown": np.nan,
            "Calmar": np.nan,
            "CAGR": np.nan,
            "Volatility": np.nan,
            "TotalReturn": np.nan
        }

    ret = eq.pct_change().dropna()
    if len(ret) == 0:
        return {
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "MaxDrawdown": np.nan,
            "Calmar": np.nan,
            "CAGR": np.nan,
            "Volatility": np.nan,
            "TotalReturn": np.nan
        }

    rf_per_period = (1 + rf) ** (1 / periods_per_year) - 1
    excess_ret = ret - rf_per_period

    mean_excess = excess_ret.mean()
    vol = ret.std(ddof=1)
    downside = excess_ret[excess_ret < 0]
    downside_vol = downside.std(ddof=1)

    sharpe = (mean_excess / vol) * np.sqrt(periods_per_year) if vol > 0 else np.nan
    sortino = (mean_excess / downside_vol) * np.sqrt(periods_per_year) if downside_vol > 0 else np.nan

    cummax = eq.cummax()
    drawdown = eq / cummax - 1
    max_dd = drawdown.min()

    total_return = eq.iloc[-1] / eq.iloc[0] - 1
    years = len(ret) / periods_per_year
    cagr = (eq.iloc[-1] / eq.iloc[0]) ** (1 / years) - 1 if years > 0 else np.nan

    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan
    ann_vol = vol * np.sqrt(periods_per_year)

    return {
        "Sharpe": sharpe,
        "Sortino": sortino,
        "MaxDrawdown": max_dd,
        "Calmar": calmar,
        "CAGR": cagr,
        "Volatility": ann_vol,
        "TotalReturn": total_return
    }

In [50]:
import pandas as pd

rows = []

for y in [2022, 2023, 2024]:
    for q in ["q1","q2","q3","q4"]:
        for model in models_to_test:
            var_name = f"{model}_{str(y)[2:]}_{q}"
            
            if var_name in globals():
                data = globals()[var_name]
                
                try:
                    equity = data["equity"]
                except:
                    equity = [x[0] for x in data]
                
                stats = perf_stats(equity)
                stats["year"] = y
                stats["quarter"] = q
                stats["model"] = model
                
                rows.append(stats)

df_metrics = pd.DataFrame(rows)


In [51]:
metric_tables = {}

for metric in ["Sharpe", "Sortino", "Calmar", "MaxDrawdown", "TotalReturn"]:
    metric_tables[metric] = (
        df_metrics
        .pivot(index="model", columns=["year", "quarter"], values=metric)
        .round(2)
    )

metric_tables

{'Sharpe': year          2022                    2023                    2024        \
 quarter         q1    q2    q3    q4    q1    q2    q3    q4    q1    q2   
 model                                                                      
 egarch        1.67  1.33  0.63 -0.03 -0.63  2.12  1.58  2.15  3.19  2.75   
 garch         1.73  1.44  0.63  0.24 -0.22  2.13  1.68  1.98  2.91  2.52   
 gjr           1.67  1.33  0.63 -0.03 -0.63  2.12  1.58  2.15  3.19  2.75   
 midas         1.91  1.50  0.61  0.00 -0.36  1.70  1.70  2.26  2.97  2.75   
 ms_LSTM_noIV  1.90  1.36  0.92  0.17  0.07  2.14  1.70  1.73  1.81  2.55   
 ms_LSTM_wIV   1.47  1.55  0.46  0.44  1.09  1.87  2.63  1.92  2.48  2.51   
 ms_base       1.52  1.63  0.47  0.18 -0.30  2.13  1.59  1.70  3.21  2.55   
 sv            1.54  0.59  0.53  0.90 -0.30  1.87  0.94  0.60  2.74  2.12   
 tarch         1.67  1.33  0.63 -0.03 -0.63  2.12  1.58  2.15  3.19  2.75   
 xg_boost      1.67  1.33  0.63  0.14 -0.63  2.12  1.58  2.15  3.1

In [52]:
with open("sharpe_table.tex", "w") as f:
    f.write(metric_tables["Sharpe"].to_latex(float_format="%.2f"))

In [53]:
df_metrics_cum = (
    df_metrics
    .groupby(["model"], as_index=False)[["Sharpe", "Sortino", "Calmar", "MaxDrawdown", "TotalReturn"]]
    .mean()
    .round(2)
)

df_metrics_cum

,model,Sharpe,Sortino,Calmar,MaxDrawdown,TotalReturn
0,egarch,1.64,1.76,32.02,-0.44,0.56
1,garch,1.66,2.04,23.83,-0.44,0.56
2,gjr,1.64,1.76,32.02,-0.44,0.56
3,midas,1.66,1.79,26.56,-0.42,0.58
4,ms_LSTM_noIV,1.21,1.23,12.96,-0.47,0.49
5,ms_LSTM_wIV,1.82,2.03,25.59,-0.44,0.67
6,ms_base,1.33,1.35,28.99,-0.44,0.56
7,sv,1.08,1.06,16.46,-0.50,0.37
8,tarch,1.64,1.76,32.02,-0.44,0.56
9,xg_boost,1.66,1.77,32.04,-0.44,0.56


In [9]:
with open("cumulative_metrics_table.tex", "w") as f:
    f.write(df_metrics_cum.to_latex(index=False, float_format="%.2f"))

### Load Saved MSGarch Outputs (NEXT : Main Index)

In [24]:

#LOAD BASE 
import pickle
globals().update(pickle.load(open("../model_outputs_19_22/ms_garch_base_outputs.pkl","rb")))


In [25]:
# [k for k in globals() if not k.startswith("__") and not k.startswith("_") and not k.startswith("get") and not k.startswith("exit")
#  and not k.startswith("pickle") and not k.startswith("In") and not k.startswith("Out")]

In [26]:

#LOAD BASE 
import pickle
globals().update(pickle.load(open("../model_outputs_19_22/ms_garch_gatedIV_outputs.pkl","rb")))


In [27]:
# [k for k in globals() if not k.startswith("__") and not k.startswith("_") and not k.startswith("get") and not k.startswith("exit")
#  and not k.startswith("pickle") and not k.startswith("In") and not k.startswith("Out")]

In [28]:

#LOAD BASE 
import pickle
globals().update(pickle.load(open("../model_outputs_19_22/ms_garch_noIV_outputs.pkl","rb")))

In [29]:
# [k for k in globals() if not k.startswith("__") and not k.startswith("_") and not k.startswith("get") and not k.startswith("exit")
#  and not k.startswith("pickle") and not k.startswith("In") and not k.startswith("Out")]

In [10]:
[k for k in globals() if k.startswith("h_final_noIV_by_year")]

['h_final_noIV_by_year']

In [ ]:
Xq_by_year[2019]

{'q1':                                  rt      bsIV       MFIV
 date_time                                               
 2019-01-01 00:00:00+00:00  0.000135  5.000000  99.647859
 2019-01-01 01:00:00+00:00 -0.004202  5.000000  99.647859
 2019-01-01 02:00:00+00:00  0.000000  5.000000  99.647859
 2019-01-01 03:00:00+00:00  0.002035  5.000000  99.647859
 2019-01-01 04:00:00+00:00  0.000271  5.000000  99.647859
 ...                             ...       ...        ...
 2019-03-31 19:00:00+00:00  0.000489  0.000001  10.119406
 2019-03-31 20:00:00+00:00 -0.000183  0.000001   9.874937
 2019-03-31 21:00:00+00:00 -0.001528  0.000001   9.636373
 2019-03-31 22:00:00+00:00  0.001345  0.000001   9.403573
 2019-03-31 23:00:00+00:00  0.000061  0.000001   9.176397
 
 [2160 rows x 3 columns],
 'q2':                                  rt      bsIV       MFIV
 date_time                                               
 2019-04-01 00:00:00+00:00  0.004450  0.000001   8.954709
 2019-04-01 01:00:00+00:00  0.00

### Load Output vs Realized

In [31]:
import pickle
outputs_by_year = pickle.load(open("../outputs_by_year/2019_2022/outputs_by_year_12mdls.pkl", "rb"))

### Create out_full

### Strategy 1

* Signal uses forecast vs realized volatility with **60-point history** and a **3-point rolling mean**.
* **Long Straddle**: top 20% signal at 15:00–16:00, buy ATM call + put (1–3 day expiry).
* **Short Strangle**: bottom 20% signal at 11:00–15:00, sell OTM call + put (~3-day expiry).
* Exit after **1 day**, on forecast extremes, after 15:00 next day, or near expiry.
* One 2-leg trade at a time, max one entry per day.
